# 08 — Season Trend Analysis

## Business Question
Did the meta shift during Season 9 (June–September 2017)? Are any patterns visible in how the game was played over time?

## What This Covers
- Weekly win rate stability (checking for data bias)
- Objective priority trends over the season
- Champion pick rate evolution
- Game duration shifts (is snowballing getting worse over patches?)

In [3]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from config import *
from data_loader import load_matches, load_champion_map
from plot_utils import set_style, save_plot

set_style()
df = load_matches()
champ_map = load_champion_map()
df['week_num'] = (df['match_date'] - df['match_date'].min()).dt.days // 7
print(f"Weeks in dataset: {df['week_num'].nunique()}")
print(f"Date range: {df['match_date'].min().date()} to {df['match_date'].max().date()}")

Weeks in dataset: 13
Date range: 2017-06-08 to 2017-09-06


In [4]:
# Weekly trends
weekly = df.groupby('week_num').agg(
    games=('t1_won', 'count'),
    t1_win_rate=('t1_won', 'mean'),
    avg_duration=('game_duration_min', 'mean'),
    first_baron_rate=('firstBaron', lambda x: (x != 0).mean()),
    first_dragon_rate=('firstDragon', lambda x: (x != 0).mean()),
    avg_baron_kills=('t1_baronKills', 'mean'),
    avg_dragon_kills=('t1_dragonKills', 'mean'),
).reset_index()
weekly = weekly[weekly['games'] >= 100]
weekly['t1_win_rate_pct'] = weekly['t1_win_rate'] * 100

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Win rate over season
axes[0,0].plot(weekly['week_num'], weekly['t1_win_rate_pct'],
               color=COLORS['blue'], linewidth=2, marker='o', markersize=5)
axes[0,0].fill_between(weekly['week_num'],
    weekly['t1_win_rate_pct'] - 2, weekly['t1_win_rate_pct'] + 2,
    alpha=0.2, color=COLORS['blue'])
axes[0,0].axhline(50, color=COLORS['gray'], linestyle='--', linewidth=1.5)
axes[0,0].set_xlabel('Week of Season')
axes[0,0].set_ylabel('Team 1 Win Rate (%)')
axes[0,0].set_title('Team 1 Win Rate by Week\n(Should be ~50% — checks for dataset bias)')
axes[0,0].set_ylim(44, 56)

# Average duration over season
axes[0,1].plot(weekly['week_num'], weekly['avg_duration'],
               color=COLORS['purple'], linewidth=2, marker='o', markersize=5)
# Trend line
z = np.polyfit(weekly['week_num'], weekly['avg_duration'], 1)
p = np.poly1d(z)
axes[0,1].plot(weekly['week_num'], p(weekly['week_num']),
               color=COLORS['red'], linestyle='--', linewidth=2, label=f'Trend (slope={z[0]:+.3f} min/week)')
axes[0,1].set_xlabel('Week of Season')
axes[0,1].set_ylabel('Avg Game Duration (minutes)')
axes[0,1].set_title('Average Duration by Week\n(Downward trend = snowballing increasing)')
axes[0,1].legend()

# Baron/Dragon objective rates over season
axes[1,0].plot(weekly['week_num'], weekly['first_baron_rate']*100,
               color=COLORS['red'], linewidth=2, label='First Baron secured rate')
axes[1,0].plot(weekly['week_num'], weekly['first_dragon_rate']*100,
               color=COLORS['orange'], linewidth=2, label='First Dragon secured rate')
axes[1,0].set_xlabel('Week of Season')
axes[1,0].set_ylabel('% of Games Where Objective Was Contested')
axes[1,0].set_title('Objective Contest Rate by Week')
axes[1,0].legend()

# Match volume over season
axes[1,1].bar(weekly['week_num'], weekly['games'],
              color=COLORS['blue'], edgecolor='white', alpha=0.85)
axes[1,1].set_xlabel('Week of Season')
axes[1,1].set_ylabel('Number of Matches')
axes[1,1].set_title('Match Volume by Week\n(Data density check)')

plt.suptitle('Season 9 Meta Trend Analysis', fontsize=14, fontweight='bold')
save_plot('08_season_trends.png')
plt.show()

# Spearman correlation: does duration trend over time?
corr, p = spearmanr(weekly['week_num'], weekly['avg_duration'])
print(f"\nDuration trend over season: rho={corr:.3f}, p={p:.4f}")
print(f"Conclusion: {'Significant' if p < 0.05 else 'No significant'} trend in game duration over the season")

  Saved -> plots/08_season_trends.png

Duration trend over season: rho=-0.308, p=0.3064
Conclusion: No significant trend in game duration over the season


## Summary

This temporal analysis checks:
1. **Data integrity** — win rates should stay near 50% throughout the season, not drift (which would indicate sampling bias)
2. **Meta shifts** — if game duration consistently decreases, snowballing is getting worse patch over patch
3. **Objective priority** — did teams shift from dragon-focus to baron-focus over the season?

These patterns matter for game analysts whose job is to detect meta health issues between patches and flag them for the balance team before they become widely problematic.